In [ ]:
import numpy as np
import matplotlib.pylab as plt

import time
import torch
import torch.nn as nn
from torch.optim import Adam

from simulation import RodentMotionModel, SquareArena, SimulationController
from networks import ContinuousTimeRNN
from regularization import LinearLambdaScheduler, AdaptiveLambdaScheduler
from regularization import (
    mean_squared_error,
    l2_weight_regularization,
    firing_rate_regularization,
)
from utils import (
    plot_square,
    plot_simulation,
    generate_dataset,
    generate_dataset_fast,
    compute_spatial_bounds,
    compute_unit_activity_map,
    compute_spatial_bin_edges_from_bounds,
    plot_activity_map
)

In [ ]:
seed = 42
theta_sigma = 0.1*2
p_move = 0.05
v_max = 0.2
arena_size = 2
max_tries = 100

In [ ]:
rodent = RodentMotionModel(seed, theta_sigma, p_move, v_max)
arena = SquareArena(arena_size)

controller = SimulationController(rodent, arena, max_tries)

In [ ]:
simulation_data = controller.generate_fast(500)

plot_simulation(simulation_data, arena)

In [ ]:
    plt.plot(
        [record["x"] for record in simulation_data]
        #[record["speed"] for record in simulation_data]
        #y=[record["y"] for record in simulation_data],
    )

In [ ]:
num_inputs = 2
num_outputs = 2
num_units = 100

noise_std = 0.01
time_constant = 0.1
time_step = time_constant / 10

In [ ]:
model = ContinuousTimeRNN(
    num_inputs,
    num_outputs,
    num_units,
    noise_std,
    time_step,
    time_constant
)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

num_steps = 500

num_epochs = 500
batches_per_epoch = 1
batch_size = 500
learning_rate = 1e-03

In [ ]:
#optimizer = Adam(model.parameters(), lr=learning_rate)
optimizer = torch.optim.NAdam(model.parameters(), lr=learning_rate, momentum_decay=1e-3, foreach=True)

weight_l2_lambda_scheduler = LinearLambdaScheduler(
    1, 0.0001, num_epochs
)
firing_rate_lambda_scheduler = AdaptiveLambdaScheduler(
    0.1, 0.3, num_epochs
)

In [ ]:
tmp = time.time()
inputs, targets = generate_dataset_fast(
        controller, 1000, num_steps, num_inputs, num_outputs
    )
print(f"Datagen time: {time.time()-tmp:.4f}s")

tmp = time.time()
inputs, targets = generate_dataset(
        controller, 1000, num_steps, num_inputs, num_outputs
    )
print(f"Datagen old time: {time.time()-tmp:.4f}s")

In [ ]:
torch.manual_seed(seed)

losses = []

model.to(device)
model._init_weights(rec_init="orthogonal")
model.train()

in_weights = model.input_weights.weight
out_weights = model.output_weights.weight

for epoch in range(1, num_epochs + 1):
    #tmp = time.time()
    inputs, targets = generate_dataset_fast(
        controller, batch_size, num_steps, num_inputs, num_outputs
    )

    inputs = inputs.to(device)
    targets = targets.to(device)
    #print(f"Datagen time: {time.time()-tmp:.4f}s")

    activities, predictions = model(inputs)

    prediction_loss = mean_squared_error(targets, predictions)
    w_regularization = l2_weight_regularization(in_weights, out_weights)
    l_w = weight_l2_lambda_scheduler.lambda_at(epoch)
            
    fr_regularization = firing_rate_regularization(activities)
    l_fr = firing_rate_lambda_scheduler.lambda_at(
        epoch, prediction_loss, fr_regularization
    )

    loss = (
        prediction_loss + l_w * w_regularization + l_fr * fr_regularization
    )
    
    optimizer.zero_grad()
    loss.backward()

    nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    losses.append(prediction_loss.item())
    print(f"Epoch: {epoch} | Loss: {prediction_loss.item():.6f}")

In [ ]:
plt.plot(losses);

In [ ]:
test_num_steps = 500
test_batches = 10
test_batch_size = 256

accumulated_mse = 0.0

model.to(device)
model.eval()

with torch.inference_mode():
    for _ in range(test_batches):
        inputs, targets = generate_dataset(
            controller, test_batch_size, test_num_steps, num_inputs, num_outputs
        )

        inputs = inputs.to(device)
        targets = targets.to(device)

        activities, predictions = model(inputs)
        accumulated_mse += mean_squared_error(targets, predictions).item()

avearge_mse = accumulated_mse / test_batches
print(f"Test MSE: {avearge_mse:.6f}")

In [ ]:
idx = np.random.randint(low=0, high=255)
print(f"idx: {idx}")

plt.plot(
    targets[idx, :, 0].detach().to("cpu"),
    targets[idx, :, 1].detach().to("cpu")
)
plt.plot(
    predictions[idx, :, 0].detach().to("cpu"),
    predictions[idx, :, 1].detach().to("cpu"), "--"
)

In [ ]:
# idx = np.random.randint(low=0, high=255)
# print(f"idx: {idx}")

plt.plot(targets[idx, :, 0].detach().to("cpu"))
plt.plot(predictions[idx, :, 0].detach().to("cpu"), "--")

In [ ]:
rodent = RodentMotionModel(seed, theta_sigma=1, p_move=1, v_max=v_max)
controller = SimulationController(rodent, arena, max_tries)

inputs, targets = generate_dataset_fast(controller, 1,  int(100000 * arena_size), 2, 2)
inputs = inputs.to(device)

activities, predictions = model(inputs)

In [ ]:
t = targets.to("cpu").detach().numpy().reshape(-1, 2)
a = activities.to("cpu").detach().numpy().reshape(-1, 100)

plot_square(size=arena.size, center=arena.center())
plt.gca().set_aspect("equal", adjustable="box")
plt.scatter(x = t[:, 0], y = t[:, 1], alpha=0.005);

In [ ]:
arena_size = arena.size
arena_center = arena.center()

positions = targets.squeeze().to("cpu").detach().numpy().reshape(-1, 2)

random_unit_id = np.random.randint(low=0, high=99)
unit_activity = activities.reshape(-1, 100)[:, random_unit_id].to("cpu").detach().numpy()

num_bins = 20

spatial_bounds = compute_spatial_bounds(
    arena_center, arena_size
)
activity_map, occupancy_map = compute_unit_activity_map(
    positions, unit_activity, spatial_bounds, num_bins
)
x_edges, y_edges = compute_spatial_bin_edges_from_bounds(
    spatial_bounds, num_bins
)

plot_activity_map(activity_map, x_edges, y_edges, title=f"unit id: {random_unit_id}")

In [ ]:
num_bins = 20

spatial_bounds = compute_spatial_bounds(arena_center, arena_size)
x_edges, y_edges = compute_spatial_bin_edges_from_bounds(spatial_bounds, num_bins)

activities_np = activities.reshape(-1, 100).to("cpu").detach().numpy()

all_maps = []
vmin = np.inf
vmax = -np.inf

for unit_id in range(activities_np.shape[1]):
    unit_activity = activities_np[:, unit_id]
    unit_activity = np.asarray(unit_activity).ravel()

    activity_map, occupancy_map = compute_unit_activity_map(
        positions, unit_activity, spatial_bounds, num_bins
    )

    all_maps.append(activity_map)
    vmin = min(vmin, np.nanmin(activity_map))
    vmax = max(vmax, np.nanmax(activity_map))

fig, axes = plt.subplots(10, 10, figsize=(16, 16), sharex=True, sharey=True)
axes = axes.ravel()

mappable = None
for unit_id, ax in enumerate(axes):
    m = ax.pcolormesh(
        x_edges, y_edges, all_maps[unit_id],
        shading="flat",
    )
    mappable = m
    ax.set_aspect("equal", adjustable="box")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(str(unit_id), fontsize=8)


plt.tight_layout()
plt.show()